## 例題
以下の要件のパイプライン・ジョブを作成して下さい。  
- パイプライン名:Delta Live Tables
- パイプライン作成場所:各自自身のuserフォルダ直下
- 作成パイプライン:ストリーミングテーブル2つ、マテアライズドビュー1つ
- 作成ジョブ:上記パイプラインを日次で一回処理されるジョブ
![](スクリーンショット 2026-03-22 21.01.44.png)


### 1. sample_blonze
- csv/DLTフォルダ内のsample.csvを取り込み
![image_1774181542780.png](./image_1774181542780.png "image_1774181542780.png")
### 2. sample_sylber
- 制約:idとnameはnot null、ageは0より上
![image_1774181623148.png](./image_1774181623148.png "image_1774181623148.png")
### 3. sample_gold
- 30歳以下、かつactionで集約し、各action項目が何回されているかをcount
![image_1774181814905.png](./image_1774181814905.png "image_1774181814905.png")



## 回答

### ①パイプラインの作成


### ②パイプラインの開発

パイプライン上の各処理のすべてに最初に入れる処理

In [0]:
%sql
USE CATALOG `workspace`;
USE SCHEMA `work`;

1つ目のストリーミングテーブルの作成

In [0]:
%sql
-- Define a streaming table to ingest data from a volume
CREATE OR REFRESH STREAMING LIVE TABLE sample_blonze
COMMENT "Sample raw data."
AS SELECT *
FROM STREAM read_files(
'/Workspace/Users/kenta.owada@jp.nttdata.com/Databricks-learn/csv/DLT')

2つ目のストリーミングテーブルの作成

In [0]:
%sql
-- Define a materialized view that validates data and renames a column
CREATE OR REFRESH STREAMING LIVE TABLE sample_sylber(
CONSTRAINT id EXPECT (id IS NOT NULL),
CONSTRAINT name EXPECT (name IS NOT NULL),
CONSTRAINT age EXPECT (age > 0)
)
COMMENT "Sample prepared data."
AS SELECT id, name, age, action
FROM STREAM(LIVE.sample_blonze)

3つ目のマテアライズドビューの作成

In [0]:
%sql
-- Define a materialized view that has a filtered, aggregated, and sorted view of the data
CREATE OR REFRESH MATERIALIZED VIEW sample_gold
COMMENT "Sample DM data."
AS SELECT
 action,
 count(*) AS count
FROM sample_sylber
WHERE age <= 30
GROUP BY action

### ③ジョブの作成